In [ ]:
"""eod_sale_service — Bronze ingest.

Real path (needs a `mongo-conn` secret): read the 2 report.* collections windowed on
transactionTime. Because the POS stores VN-LOCAL time, the window is the VN calendar day
[running_date 00:00, +1d) — NO UTC shift (contrast eod_sale_product, whose documentDate
is UTC). Nested items/orderInfo kept intact. NOTE: top_up_transaction carries both Top Up
and Pay Card docs (silver splits them by serviceName).

POC / DEV path: bronze is seeded by simulators_service.save_service_bronze (or the
pipeline's Copy activities), so this is skipped and the pipeline runs with with_ingest=False.
"""
import json

from ssv_data.io.readers import Readers
from ssv_data.io.writers import write_delta

SERVICE_COLLECTIONS = ["pay_bill_transaction", "top_up_transaction"]


In [ ]:
def _opt(ctx, key):
    try:
        return ctx.secret(key)
    except Exception:
        return None


def _local_day_window(ctx):
    """$match on transactionTime for the VN calendar day (source time is already local).
    Full ISO with millis + Z so the bson extended-JSON date parser accepts it."""
    import datetime as _dt
    lo = ctx.running_date + "T00:00:00.000Z"
    hi = ((_dt.datetime.strptime(ctx.running_date, "%Y-%m-%d") + _dt.timedelta(days=1))
          .strftime("%Y-%m-%dT00:00:00.000Z"))
    return json.dumps([{"$match": {"transactionTime": {"$gte": {"$date": lo}, "$lt": {"$date": hi}}}}])


def service_bronze_ingest(ctx) -> None:
    if not _opt(ctx, "mongo-conn"):
        ctx.logger.info("bronze(service): Mongo SKIPPED (no mongo-conn) — bronze expected "
                        "already present (simulator seed or Copy activity)")
        return
    r = Readers(ctx)
    pl = _local_day_window(ctx)
    for coll in SERVICE_COLLECTIONS:
        write_delta(ctx, r.mongo("report", coll, pl), ctx.bronze(coll))
    ctx.logger.info(f"bronze(service): Mongo ingested {SERVICE_COLLECTIONS}")
